In [9]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [10]:
from getpass import getpass
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [11]:
import re

## Deterministic approach
def deterministic_approach(text: str) -> bool:
    """Return True when the content is blocked."""
    banned_keywords = ["hack", "malware", "virus", "exploit", "attack", "bomb"]
    return any(keyword in text.lower() for keyword in banned_keywords)


test_input = [
    "How to hack a computer system",
    "How to make a cake",
    "How to make a bomb",
]

print("Deterministic approach results:")
for inp in test_input:
    if deterministic_approach(inp):
        print(f"blocked: {inp}")
    else:
        print(f"allowed: {inp}")

Deterministic approach results:
blocked: How to hack a computer system
allowed: How to make a cake
blocked: How to make a bomb


In [12]:
### model based approach

from langchain_openai import ChatOpenAI


def model_based_approach(text: str) -> str:
    """Use the LLM to evaluate the content and return safe or unsafe."""
    llm = ChatOpenAI(model="gpt-4", temperature=0)
    prompt = (
        "Classify the following user input as exactly one word: safe or unsafe. "
        "Return only that word. Treat requests for harmful instructions as unsafe. "
        f"Input: {text}"
    )
    result = llm.invoke(prompt)
    return str(result.content).strip().lower()


print("model based approach results:")
for inp in test_input:
    result = model_based_approach(inp)
    status = "unsafe" if result == "unsafe" else "safe"
    print(f"{status}: {inp}")

model based approach results:
unsafe: How to hack a computer system
safe: How to make a cake
unsafe: How to make a bomb


In [14]:
### PII approach

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


## Define a simple dummy tool
@tool
def dummy_tool(query: str) -> str:
    """Lookup customer information."""
    return f"Customer record found for query: {query}"


model = ChatOpenAI(model="gpt-4", temperature=0)

### Agent for redaction and masking
pii_agent = create_agent(
    model=model,
    tools=[dummy_tool],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        
        ),


        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{48,}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("PII-protected agents created successfully.")

PII-protected agents created successfully.


In [15]:
### test PII redaction and masking
results=pii_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Use the customer lookup tool for john.doe@example.com and card 4111-1111-1111-1111.can you help me?"
    }]
})

print("agent response:")
print(results["messages"][-1].content)



agent response:
I found the customer record for the email and card you provided. How can I assist you further?


In [16]:
results

{'messages': [HumanMessage(content='Use the customer lookup tool for [REDACTED_EMAIL] and card ****-****-****-1111.can you help me?', additional_kwargs={}, response_metadata={}, id='bb77070e-8de2-478b-8b59-1d38a5a466ce'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 71, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-ETvScsEtpzD08B6f20zeXExD7S0XB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f410-3200-7f53-8417-4f2c093e5b04-0', tool_calls=[{'name': 'dummy_tool', 'args': {'query': '[REDACTED_EMAIL] and card ****-****-****-1111'}, 'id': 'call_SldF